In [ ]:
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import randint, uniform

# … 之前的导入和数据准备省略 …

# 定义随机森林和参数分布
rf = RandomForestRegressor(random_state=0, n_jobs=-1)
param_dist = {
    'n_estimators': randint(100, 1000),
    'max_depth': randint(5, 30),
    'max_features': uniform(0.3, 0.7),
    'min_samples_split': randint(2, 20),
    'bootstrap': [True, False],
}

# 在循环里替换
search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=200,
    scoring='r2',
    cv=cv,
    verbose=2,
    n_jobs=-1,
    random_state=0
)

search.fit(X_train, y_train)
best_model = search.best_estimator_

# 评估
y_test_pred = best_model.predict(X_test)
r2_test = r2_score(y_test, y_test_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
print(f"✅ {filename} | {target} | RF最佳参数: {search.best_params_} | R²_test={r2_test:.3f} | RMSE_test={rmse_test:.3f}")

# 特征重要性
for var, importance in zip(explanatory_vars, best_model.feature_importances_):
    all_results.append({
        'GridSize': grid_size,
        'Target': target,
        'Feature': var,
        'FeatureImportance_TrainModel': round(importance, 4),
        # … 其它指标 …
        **search.best_params_
    })

# PDP 画法同 GBDT
for feature in top_features:
    fig, ax = plt.subplots()
    disp = PartialDependenceDisplay.from_estimator(best_model, X, [feature], ax=ax)
    x_vals = disp.lines_[0][0].get_xdata()
    y_vals = disp.lines_[0][0].get_ydata()
    plt.close(fig)
    pdp_records.append({
        'Feature': feature,
        'GridSize': grid_size,
        'Target': target,
        'X': x_vals,
        'Y': y_vals
    })
